# VBZ Geo-Map — lonboard

GPU-beschleunigte Kartenvisualisierung mit **lonboard** (Deck.gl).
Moderne Alternative zu Kepler.gl — Python 3.13 kompatibel.

```bash
pip install lonboard
```

---

## Architektur-Notizen

**Stärken:**
- GPU-Rendering via Deck.gl: tausende Segmente ohne Performance-Probleme
- Python 3.13 kompatibel (keplergl funktioniert nicht mit Python 3.13)
- HeatmapLayer, PathLayer, ScatterplotLayer, ArcLayer out of the box
- Farben per Colormap auf numpy-Arrays — kein Trace-Limit wie bei Plotly

**Einschränkungen:**
- Kein VS Code Rendering — braucht JupyterLab
- Weniger Dashboard-Integration als Plotly

**Empfehlung:** Explorative Tiefenanalyse und Hotspot-Suche im Gesamtnetz.


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
from pathlib import Path
import matplotlib.cm as mcm

from lonboard import Map, PathLayer, ScatterplotLayer, PolygonLayer, HeatmapLayer
from lonboard.view_state import MapViewState
from lonboard.basemap import CartoBasemap
from lonboard.colormap import apply_continuous_cmap

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        _ROOT = _p
        break

GTFS_DIR = _ROOT / 'data' / 'interim' / 'gtfs'
GEO_DIR  = _ROOT / 'data' / 'raw' / 'geo' / 'data'

routes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_routes.parquet')
stops  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_stops.parquet')
shapes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_shapes.parquet')
trips  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_trips.parquet')

routes_2024  = routes[routes['year'] == '2024'].copy()
shapes_2024  = shapes[shapes['year'] == '2024'].copy()
stops_2024   = stops[stops['year'] == '2024'].copy()
trips_2024   = trips[trips['year'] == '2024'].copy()
stops_unique = stops_2024.drop_duplicates(subset=['stop_name']).copy()

np.random.seed(42)
stops_unique['delay_min'] = np.random.exponential(scale=1.5, size=len(stops_unique)).round(1)

gdf_kreise = gpd.read_file(GEO_DIR / 'stzh_adm_stadtkreise_v.json')

def best_shape_for_line(line_name):
    route_ids = routes_2024[routes_2024['route_short_name'] == line_name]['route_id']
    shape_ids = trips_2024[trips_2024['route_id'].isin(route_ids)]['shape_id'].unique()
    if len(shape_ids) == 0:
        return None
    return max(shape_ids, key=lambda s: len(shapes_2024[shapes_2024['shape_id'] == s]))

def hex_to_rgba(hex_color, alpha=200):
    h = hex_color.lstrip('#')
    return [int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), alpha]

DARK = CartoBasemap.DarkMatter
print(f'Stops: {len(stops_unique)}  Linien: {len(routes_2024)}')

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        REPORTS_DIR = _p / 'assets'
        break
REPORTS_DIR.mkdir(exist_ok=True)


Stops: 1189  Linien: 16


## Karte 1 — Stadtkreise + Tramnetz + Haltestellen


In [2]:
layer_kreise = PolygonLayer.from_geopandas(
    gdf_kreise,
    get_fill_color=[42, 42, 78, 80],
    get_line_color=[170, 170, 200, 180],
    get_line_width=30,
    stroked=True, filled=True,
)

line_rows = []
for line in sorted(routes_2024['route_short_name'].unique()):
    shape_id = best_shape_for_line(line)
    if shape_id is None:
        continue
    pts = shapes_2024[shapes_2024['shape_id'] == shape_id].sort_values('shape_pt_sequence')
    color_hex = routes_2024[routes_2024['route_short_name'] == line]['route_color'].iloc[0]
    line_rows.append({'geometry': LineString(zip(pts['shape_pt_lon'], pts['shape_pt_lat'])),
                      'color': hex_to_rgba(color_hex)})

gdf_lines = gpd.GeoDataFrame(line_rows, crs='EPSG:4326')
colors_lines = np.array([r['color'] for r in line_rows], dtype=np.uint8)

layer_lines = PathLayer.from_geopandas(
    gdf_lines, get_color=colors_lines, get_width=15, width_min_pixels=2,
)

gdf_stops = gpd.GeoDataFrame(
    stops_unique,
    geometry=gpd.points_from_xy(stops_unique['stop_lon'], stops_unique['stop_lat']),
    crs='EPSG:4326'
)
layer_stops = ScatterplotLayer.from_geopandas(
    gdf_stops, get_fill_color=[255, 255, 255, 180],
    get_radius=30, radius_min_pixels=2,
)

Map(layers=[layer_kreise, layer_lines, layer_stops], basemap_style=DARK,
    view_state=MapViewState(latitude=47.378, longitude=8.540, zoom=12))


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_33492/3225185744.py:36: DeprecationWarning: `basemap_style` is deprecated and will be removed in 0.14. Use `basemap` instead.
  Map(layers=[layer_kreise, layer_lines, layer_stops], basemap_style=DARK,


## Karte 2 — Verspätungs-Heatmap


In [3]:
gdf_stops_delay = gpd.GeoDataFrame(
    stops_unique,
    geometry=gpd.points_from_xy(stops_unique['stop_lon'], stops_unique['stop_lat']),
    crs='EPSG:4326'
)

layer_kreise_outline = PolygonLayer.from_geopandas(
    gdf_kreise,
    get_fill_color=[0, 0, 0, 0],
    get_line_color=[170, 170, 200, 150],
    get_line_width=30,
    stroked=True, filled=False,
)

layer_heat = HeatmapLayer.from_geopandas(
    gdf_stops_delay,
    get_weight=stops_unique['delay_min'].values,
    radius_pixels=40,
)

Map(layers=[layer_kreise_outline, layer_heat], basemap_style=DARK,
    view_state=MapViewState(latitude=47.378, longitude=8.540, zoom=12))


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_33492/1333246723.py:21: DeprecationWarning: `basemap_style` is deprecated and will be removed in 0.14. Use `basemap` instead.
  Map(layers=[layer_kreise_outline, layer_heat], basemap_style=DARK,


## Karte 2b — Haltestellen eingefärbt nach Verspätung


In [4]:
max_delay = stops_unique['delay_min'].quantile(0.95)
normalized = (stops_unique['delay_min'].clip(0, max_delay) / max_delay).values
colors_delay = apply_continuous_cmap(normalized, mcm.RdYlGn_r, alpha=0.9)

layer_stops_delay = ScatterplotLayer.from_geopandas(
    gdf_stops_delay, get_fill_color=colors_delay,
    get_radius=60, radius_min_pixels=4,
)

Map(layers=[layer_kreise_outline, layer_stops_delay], basemap_style=DARK,
    view_state=MapViewState(latitude=47.378, longitude=8.540, zoom=12))


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_33492/2981002379.py:10: DeprecationWarning: `basemap_style` is deprecated and will be removed in 0.14. Use `basemap` instead.
  Map(layers=[layer_kreise_outline, layer_stops_delay], basemap_style=DARK,


## Karte 3 — Verspätung auf Streckenabschnitten (Linie 11)

> Zwei Segmente als Beispiel — identisch mit Plotly/GeoPandas/Folium.


In [5]:
h1 = stops_2024[stops_2024['stop_name'] == 'Zürich, Paradeplatz'].iloc[0]
h2 = stops_2024[stops_2024['stop_name'] == 'Zürich, Bellevue'].iloc[0]
h3 = stops_2024[stops_2024['stop_name'] == 'Zürich, Bahnhof Stadelhofen'].iloc[0]

route_ids_11 = routes_2024[routes_2024['route_short_name'] == '11']['route_id']
shape_ids_11 = trips_2024[trips_2024['route_id'].isin(route_ids_11)]['shape_id'].unique()
shape_id_11  = max(shape_ids_11, key=lambda s: len(shapes_2024[shapes_2024['shape_id'] == s]))
shape_11     = shapes_2024[shapes_2024['shape_id'] == shape_id_11].sort_values('shape_pt_sequence')
color_hex_11 = routes_2024[routes_2024['route_short_name'] == '11']['route_color'].iloc[0]

gdf_line11 = gpd.GeoDataFrame(
    geometry=[LineString(zip(shape_11['shape_pt_lon'], shape_11['shape_pt_lat']))],
    crs='EPSG:4326'
)
layer_line11 = PathLayer.from_geopandas(
    gdf_line11, get_color=hex_to_rgba(color_hex_11, alpha=180),
    get_width=20, width_min_pixels=3,
)

gdf_segs = gpd.GeoDataFrame([
    {'geometry': LineString([(h1['stop_lon'], h1['stop_lat']), (h2['stop_lon'], h2['stop_lat'])])},
    {'geometry': LineString([(h2['stop_lon'], h2['stop_lat']), (h3['stop_lon'], h3['stop_lat'])])},
], crs='EPSG:4326')

seg_colors = np.array([[255, 215, 0, 230], [229, 57, 53, 230]], dtype=np.uint8)

layer_segs = PathLayer.from_geopandas(
    gdf_segs, get_color=seg_colors, get_width=60, width_min_pixels=6,
)

gdf_h = gpd.GeoDataFrame([
    {'geometry': Point(h1['stop_lon'], h1['stop_lat'])},
    {'geometry': Point(h2['stop_lon'], h2['stop_lat'])},
    {'geometry': Point(h3['stop_lon'], h3['stop_lat'])},
], crs='EPSG:4326')

layer_h = ScatterplotLayer.from_geopandas(
    gdf_h, get_fill_color=[255, 255, 255, 230],
    get_radius=50, radius_min_pixels=8,
)

Map(layers=[layer_kreise_outline, layer_line11, layer_segs, layer_h], basemap_style=DARK,
    view_state=MapViewState(latitude=47.3695, longitude=8.5430, zoom=14))


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_33492/2835269552.py:42: DeprecationWarning: `basemap_style` is deprecated and will be removed in 0.14. Use `basemap` instead.
  Map(layers=[layer_kreise_outline, layer_line11, layer_segs, layer_h], basemap_style=DARK,
